# 🟢 Interview: Implement Max Pool 2D

$$H_{out} = \lfloor (H - k) / s \rfloor + 1$$

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def max_pool2d(x, kernel_size, stride=None):
    """
    手写 Max Pool 2D —— 面试版。

    面试考点:
    1. unfold 是 PyTorch 提取滑动窗口的利器
    2. 如果不允许用 unfold，就需要嵌套循环（面试时可以先写循环版再优化）
    3. 输出尺寸公式: H_out = (H - k) // stride + 1

    参数: x: (B, C, H, W), kernel_size: int, stride: int or None
    返回: (B, C, H_out, W_out)
    """
    if stride is None:
        stride = kernel_size

    # ---- 方法 1: 使用 unfold (高效) ----
    # unfold(2, k, s) 在 H 维提取窗口 → (B, C, H_out, W, k)
    # unfold(3, k, s) 在 W 维提取窗口 → (B, C, H_out, W_out, k, k)
    patches = x.unfold(2, kernel_size, stride).unfold(3, kernel_size, stride)
    # flatten(-2) 合并 k*k → max(dim=-1) 取最大值
    return patches.flatten(-2).max(dim=-1).values

    # ---- 方法 2: 纯循环版 (面试时如果不用 unfold) ----
    # B, C, H, W = x.shape
    # H_out = (H - kernel_size) // stride + 1
    # W_out = (W - kernel_size) // stride + 1
    # out = torch.zeros(B, C, H_out, W_out)
    # for i in range(H_out):
    #     for j in range(W_out):
    #         h_start, w_start = i * stride, j * stride
    #         window = x[:, :, h_start:h_start+kernel_size, w_start:w_start+kernel_size]
    #         out[:, :, i, j] = window.amax(dim=(2, 3))  # 在 H,W 维取 max
    # return out

In [ ]:
import torch.nn.functional as F

x = torch.randn(1, 1, 4, 4)
out = max_pool2d(x, kernel_size=2, stride=2)
ref = F.max_pool2d(x, kernel_size=2, stride=2)
print("Output shape:", out.shape)
print("Matches F.max_pool2d:", torch.allclose(out, ref))

In [ ]:
from torch_judge import check
check("max_pool2d")

## 📝 核心思路总结

1. **unfold 是滑动窗口利器**：`x.unfold(dim, size, step)` 在指定维度上提取固定大小的窗口，自动处理边界和步长。
2. **输出尺寸公式**：`H_out = (H - k) // stride + 1`，面试必背；stride=k 时不重叠，stride<k 时有重叠。
3. **flatten + max 取池化值**：把窗口的 2D 区域展平为 1D，再取 max，等价于在 (k, k) 区域取最大值。
4. **纯循环版是备选**：如果面试官不让用 unfold，嵌套循环 + slice + amax 也能实现，只是效率低。